# 4-тапсырма — Машиналық оқыту және нейрондық желілер
**Тақырып:** шағын автономды робот бетті ТҰРАҚТЫ (1) немесе ТҰРАҚСЫЗ (0) деп жіктейді.

**Colab-та іске қосу:** *Runtime → Run all*. Қосымша орнату керек емес (scikit-learn, pandas, NumPy Colab-та бар).
Ноутбук A–F бөлімдерінің нәтижелерін шығарады. Нәтижелерді Word құжатындағы кестелермен салыстырып тексеріңіз.

## Деректер жиынтығы және алдын ала өңдеу
24 жол, 4 кіріс белгісі (Vibration, Slip, SurfaceGrip, ObstacleRate) және мақсатты айнымалы Stable.
**Бөлу:** оқыту — ID 1–18, тест — ID 19–24 (тапсырмада осылай берілген, кездейсоқ бөлу емес).

**Алдын ала өңдеу:** белгілер бір шамада (0–1), бірақ нейрондық желі мен логистикалық регрессия үшін `StandardScaler` қолданамыз (орташа 0, ауытқу 1). Масштабтау параметрлері **тек оқыту жолдарынан** есептеледі (pipeline ішінде `fit` кезінде), сондықтан тест деректерінен ақпарат "сыртқа шықпайды" (data leakage болмайды). Шешім ағашына масштабтау қажет емес.

In [ ]:
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

# ID, Vibration, Slip, SurfaceGrip, ObstacleRate, Stable (1 = тұрақты, 0 = тұрақсыз)
rows = [
 (1,0.20,0.10,0.80,0.10,1),(2,0.30,0.20,0.70,0.20,1),(3,0.40,0.15,0.90,0.15,1),
 (4,0.50,0.30,0.80,0.10,1),(5,0.60,0.25,0.75,0.20,1),(6,0.70,0.35,0.85,0.10,1),
 (7,0.80,0.40,0.90,0.15,1),(8,0.25,0.50,0.70,0.40,1),(9,0.35,0.45,0.80,0.35,1),
 (10,0.45,0.55,0.75,0.45,1),(11,0.55,0.50,0.85,0.30,1),(12,0.65,0.60,0.80,0.40,1),
 (13,0.20,0.80,0.40,0.75,0),(14,0.30,0.70,0.35,0.80,0),(15,0.40,0.85,0.45,0.70,0),
 (16,0.50,0.75,0.30,0.85,0),(17,0.60,0.90,0.40,0.80,0),(18,0.70,0.80,0.35,0.90,0),
 (19,0.80,0.85,0.45,0.75,0),(20,0.90,0.70,0.30,0.85,0),(21,0.25,0.90,0.50,0.80,0),
 (22,0.45,0.95,0.40,0.90,0),(23,0.65,0.85,0.30,0.95,0),(24,0.85,0.90,0.35,0.85,0)]
df = pd.DataFrame(rows, columns=["ID","Vibration","Slip","SurfaceGrip","ObstacleRate","Stable"])
FEATURES = ["Vibration", "Slip", "SurfaceGrip", "ObstacleRate"]

train, test = df[df.ID <= 18], df[df.ID >= 19]
X_train, y_train = train[FEATURES].values, train.Stable.values
X_test,  y_test  = test[FEATURES].values,  test.Stable.values
print(df.shape, "| оқыту:", X_train.shape, "| тест:", X_test.shape)

## A. Деректер жиынтығын сипаттау
Мұнда жолдар саны, белгілер саны, кластар, класс балансы және кластар арасындағы орташа айырмашылықтар есептеледі.
Ең үлкен айырмашылығы бар белгі — кластардың орташа мәндерінің айырмасы ең үлкен белгі.
**Маңызды бақылау:** тест жиынында тек Тұрақсыз (0) үлгілер бар — бұл метрикаларды түсіндіруге қатты әсер етеді.

In [ ]:
print("Оқыту / тест жолдары:", len(train), "/", len(test))
print("Кіріс белгілерінің саны:", len(FEATURES), FEATURES)
print("Кластар:", [int(c) for c in sorted(df.Stable.unique())], "(1 = Тұрақты, 0 = Тұрақсыз)")
print("Оқыту:  Тұрақты =", (train.Stable == 1).sum(), "| Тұрақсыз =", (train.Stable == 0).sum())
print("Тест:   Тұрақты =", (test.Stable == 1).sum(),  "| Тұрақсыз =", (test.Stable == 0).sum())
print("Барлығы: Тұрақты =", (df.Stable == 1).sum(),   "| Тұрақсыз =", (df.Stable == 0).sum())

means = train.groupby("Stable")[FEATURES].mean().round(3)
means.loc["|айырма|"] = (means.loc[1] - means.loc[0]).abs()
print("\nОқыту жиынындағы кластар бойынша орташа мәндер:")
print(means.to_string())
print("\nТұрақты кластың диапазоны:\n", train[train.Stable == 1][FEATURES].agg(["min", "max"]).to_string())
print("Тұрақсыз кластың диапазоны:\n", train[train.Stable == 0][FEATURES].agg(["min", "max"]).to_string())

## B. Базалық модельдер және метрикалар
Екі базалық модель оқытамыз: **шешім ағашы** (ереже түрінде түсінікті) және **логистикалық регрессия** (салмақтар мен ықтималдық береді).

**Метрикаларды оқу:** тапсырмада Тұрақты = 1 деп алынған, сондықтан әдепкі оң класс — Тұрақты. Бірақ тестте Тұрақты үлгі жоқ (TP + FN = 0), сондықтан Тұрақты-оң Precision/Recall/F1 **анықталмаған (0/0)**; scikit-learn `zero_division=0` кезінде 0.0 көрсетеді — бұл "модель нашар" дегенді білдірмейді. Сондықтан қауіпті класс — **Тұрақсыз** — оң класс болғандағы метрикаларды да шығарамыз.

**Сұрақ:** ең қауіпті қате — ТҰРАҚСЫЗ бетті ТҰРАҚТЫ деп болжау. Тұрақты = оң класс болса, бұл — **False Positive (FP)**, сондықтан **Precision = TP/(TP+FP)** (Тұрақты болжамдарының сенімділігі) ерекше маңызды. Баламалы көзқарас — Тұрақсыз класының **Recall**-ы (қауіпті беттердің қаншасын модель тапты) = TN/(TN+FP). Жоғары шек қою (мысалы, P(Stable) ≥ 0.8 болғанда ғана "тұрақты" деу) FP-ді азайтады.

In [ ]:
def report(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])        # [[TN, FP], [FN, TP]]
    print(f"--- {name} ---")
    print("Қателер матрицасы [[TN FP],[FN TP]]:", cm.tolist())
    print(f"Accuracy = {accuracy_score(y_true, y_pred):.3f}")
    for pos, label in [(1, "Тұрақты = оң"), (0, "Тұрақсыз = оң")]:
        P = precision_score(y_true, y_pred, pos_label=pos, zero_division=0)
        R = recall_score(y_true, y_pred, pos_label=pos, zero_division=0)
        F = f1_score(y_true, y_pred, pos_label=pos, zero_division=0)
        print(f"  {label:14s}: Precision={P:.3f}  Recall={R:.3f}  F1={F:.3f}")

tree   = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
logreg = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
pred_tree, pred_lr = tree.predict(X_test), logreg.predict(X_test)

report("Decision Tree", y_test, pred_tree)
report("Logistic Regression", y_test, pred_lr)
print("\nШешім ағашының ережесі:\n" + export_text(tree, feature_names=FEATURES))
print("LR: P(Stable) тест жолдарында:", logreg.predict_proba(X_test)[:, 1].round(3))
print("LR салмақтары (стандартталған белгілер):", {f: float(w) for f, w in zip(FEATURES, logreg[-1].coef_[0].round(3))})

## C. Нейрондық желі
**Құрылым:** кіріс (4 белгі) → бір жасырын қабат (ReLU) → шығыс (сигмоида, P(Stable)). Оптимизатор — Adam, оқыту жылдамдығы 0.001, `random_state=42` (нәтиже қайталанатындай).

**Бақыланатын эксперимент:** тек **бір гиперпараметрді** — жасырын қабаттағы нейрон санын өзгертеміз: бастапқы мән **4**, жаңа мәндер **2** және **8**. Қалған барлығы (деректер, масштабтау, жылдамдық, seed, итерация саны) өзгермейді. Сонда айырмашылық тек нейрон санынан шығады.

In [ ]:
def make_nn(hidden, seed=42):
    return make_pipeline(StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(hidden,), activation="relu", solver="adam",
                      learning_rate_init=0.001, max_iter=2000, random_state=seed))

results, nn_models = [], {}
for hidden in [4, 2, 8]:                      # 4 = бастапқы мән, 2 және 8 = жаңа мәндер
    nn = make_nn(hidden).fit(X_train, y_train)
    nn_models[hidden] = nn
    pred = nn.predict(X_test)
    proba = nn.predict_proba(X_test)[:, 1]
    results.append({"нейрон": hidden,
                    "train_acc": nn.score(X_train, y_train),
                    "test_acc": accuracy_score(y_test, pred),
                    "F1(Тұрақты=оң)": f1_score(y_test, pred, pos_label=1, zero_division=0),
                    "F1(Тұрақсыз=оң)": f1_score(y_test, pred, pos_label=0, zero_division=0),
                    "max P(Stable)": round(float(proba.max()), 3),
                    "итерация": nn[-1].n_iter_, "loss": round(nn[-1].loss_, 4)})
print(pd.DataFrame(results).to_string(index=False))

## D. Әрбір тест жағдайын (ID 19–24) тексеру
Төмендегі кестеде әр тест жолы үшін нақты класс, үш модельдің болжамы, P(Stable) және дұрыстығы көрсетіледі. Қате болса, оның себебін белгі мәндері бойынша іздейміз (мысалы, оқыту диапазонынан тыс мән).

In [ ]:
nn4 = nn_models[4]                       # бастапқы нейрондық желі (4 нейрон)
pred_nn = nn4.predict(X_test)
table = pd.DataFrame({
    "ID": test.ID.values, "Нақты": y_test,
    "DT": pred_tree, "LR": pred_lr, "NN": pred_nn,
    "P(Stable) LR": logreg.predict_proba(X_test)[:, 1].round(3),
    "P(Stable) NN": nn4.predict_proba(X_test)[:, 1].round(3)})
table["Барлығы дұрыс?"] = ((table.DT == table["Нақты"]) & (table.LR == table["Нақты"]) &
                           (table.NN == table["Нақты"])).map({True: "Иә", False: "Жоқ"})
print(table.to_string(index=False))
print("\nҚате болған жолдар:", table[table["Барлығы дұрыс?"] == "Жоқ"].ID.tolist() or "жоқ")
print("\nТест жолдарының белгілері:"); print(test.to_string(index=False))

## E. Техникалық талдау — қосымша тексерулер
1. **Seed сезімталдығы:** нейрондық желінің бастапқы салмақтары кездейсоқ. Әр нейрон саны үшін 20 түрлі `random_state` сынаймыз және тест жолдарында қате болған жағдайларды санаймыз.
2. **Бір белгілі ағаштар (ablation):** әр белгіні жеке қолданып, қайсысы кластарды өз бетінше ажырататынын тексереміз. Vibration белгісі кластарды ажыратпайды (орташа мәндері 0.479 және 0.450), сондықтан ол модельді адастыруы мүмкін.

In [ ]:
for hidden in [2, 4, 8]:
    bad = {}
    for seed in range(20):
        p = make_nn(hidden, seed).fit(X_train, y_train).predict(X_test)
        if (p != y_test).any():
            bad[seed] = [int(i) for i in test.ID.values[p != y_test]]
    print(f"нейрон={hidden}: тест жолдарында қате болған seed саны = {len(bad)}/20; қате ID-лер: {bad}")

print()
for f in FEATURES:
    t = DecisionTreeClassifier(max_depth=1, random_state=0).fit(train[[f]].values, y_train)
    print(f"{f:13s} тек осы белгімен: train acc = {t.score(train[[f]].values, y_train):.3f}, "
          f"test acc = {t.score(test[[f]].values, y_test):.3f}")

## F. Үйрену – жаттап алу – түсіну: жаттап алуды жалпылаудан ажырату эксперименті
**Эксперимент (кездейсоқ белгілер / label shuffling):** нейрондық желіні (8 нейрон) екі жағдайда оқытамыз:
(1) **нақты** белгілермен, (2) белгілері **кездейсоқ араластырылған** деректермен. Екеуінде де Leave-One-Out (LOO) арқылы әр жолды оқытуға қатыспаған күйінде тексереміз.

**Неліктен ақпаратты?** Кездейсоқ белгілерде заңдылық жоқ — оларды тек жаттап алуға болады. Егер желі кездейсоқ белгілерде оқыту дәлдігін жоғары көрсетіп (жаттап алу қабілеті), бірақ LOO дәлдігі ≈ кездейсоқ болса (50%) және нақты белгілерде LOO дәлдігі жоғары болса — демек, нақты белгілерде желі жаттап алып қана қоймай, деректегі шынайы заңдылықты жалпылаған. Бұл жаттап алу мен жалпылауды бір-бірінен ажыратады.

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_score

X_all, y_all = df[FEATURES].values, df.Stable.values
real_loo = cross_val_score(make_nn(8), X_all, y_all, cv=LeaveOneOut()).mean()

shuf_loo, shuf_train = [], []
for s in range(10):
    y_s = np.random.RandomState(s).permutation(y_all)          # белгілерді араластыру
    shuf_loo.append(cross_val_score(make_nn(8), X_all, y_s, cv=LeaveOneOut()).mean())
    shuf_train.append(make_nn(8).fit(X_all, y_s).score(X_all, y_s))

print(f"Нақты белгілер:        LOO accuracy = {real_loo:.3f}")
print(f"Араласқан белгілер:    LOO accuracy = {np.mean(shuf_loo):.3f}  (кездейсоқ деңгей ~0.5)")
print(f"Араласқан белгілер:    оқыту accuracy = {np.mean(shuf_train):.3f}  (жаттап алу мүмкіндігі)")

## Қорытынды
- Барлық модельдер тест жиынында 6/6 дұрыс болжады, бірақ тестте тек Тұрақсыз жолдар бар; сондықтан бұл нәтиже мықты дәлел емес.
- Метрикаларды Тұрақсыз = оң класс етіп те есептеген пайдалы; ең қауіпті қате (Тұрақсызды Тұрақты деп болжау) тестте 0 рет болды.
- Деректер аз (24 жол), стратификациясыз тізбекті бөлу тест нәтижесін шектейді. Келесі қадам: стратификацияланған бөлу немесе көп қайталанған cross-validation және жаңа беттер деректері.